###B1

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
df_stores_bronze = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"/Volumes/{catalog}/landing/raw/stores/")
)


In [0]:
from pyspark.sql import functions as F

(
    df_stores_bronze
    .withColumn("source_filename", F.col("_metadata.file_name"))
    .withColumn("ingested_at", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.bronze.stores")
)

###B2

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.customers")

spark.sql(f"""
COPY INTO {catalog}.bronze.customers
FROM (
  SELECT *,
    _metadata.file_path as _source_file,
    current_timestamp() as _ingested_at
  FROM '/Volumes/{catalog}/landing/raw/customers/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
  'header' = 'true',
  'inferSchema' = 'true'
)
COPY_OPTIONS (
  'mergeSchema' = 'true'
)
""")

###B3

In [0]:
%sql
create table if not exists identifier(:catalog || '.bronze.products') as 
select *,
    _metadata.file_path as _source_file,
    current_timestamp() as _ingested_at
from read_files('/Volumes/fk_ce_2026_07_1378/landing/raw/products/', format => 'json', multiLine => true)

###B4

In [0]:
catalog_name = dbutils.widgets.get("catalog")
orders_path = f"/Volumes/{catalog_name}/landing/raw/orders/"
schema_loc = f"/Volumes/{catalog_name}/ops/pipeline_state/orders_schema_v2"
checkpoint_loc = f"/Volumes/{catalog_name}/ops/pipeline_state/orders_checkpoint_v2"

orders_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_loc)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.includeExistingFiles", "true")
        .option("rescuedDataColumn", "_rescued_data")
        .load(orders_path)
)

query = (
    orders_stream.writeStream
        .option("checkpointLocation", checkpoint_loc)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .format("delta")
        .toTable(f"{catalog_name}.bronze.orders")
)

query.awaitTermination()


In [0]:
from pyspark.sql import Row, functions as F

catalog = dbutils.widgets.get("catalog")

# Get job run ID and task key from job context (available when run as a job task)
try:
    job_run_id = dbutils.widgets.get("run_id")
except Exception:
    job_run_id = "manual_run"
try:
    task_name = dbutils.widgets.get("task_key")
except Exception:
    task_name = "ingest_bronze"

# Tables written by this task (ingest_bronze / Section B)
tables_and_counts = [
    ("bronze.stores", spark.table(f"{catalog}.bronze.stores").count()),
    ("bronze.customers", spark.table(f"{catalog}.bronze.customers").count()),
    ("bronze.products", spark.table(f"{catalog}.bronze.products").count()),
    ("bronze.orders", spark.table(f"{catalog}.bronze.orders").count()),
]

audit_rows = spark.createDataFrame([
    Row(job_run_id=str(job_run_id), task_name=task_name, target_table=t[0], row_count=int(t[1]))
    for t in tables_and_counts
]).withColumn("completed_at", F.current_timestamp())

audit_rows.write.mode("append").saveAsTable(f"{catalog}.ops.load_audit")
print(f"Appended {audit_rows.count()} load_audit rows for {task_name}")